In [1]:
# =======================================================================================
#
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
#
# =======================================================================================
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import os
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize

print("Libraries imported successfully.")

# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    return np.mean(width + penalty_lower + penalty_upper)

# --- Global Constants ---
DATA_PATH = './'
PREDS_SAVE_PATH = './model_predictions/'
OLD_BEST_SCORE = 297534.88 # Your current best score to beat

Libraries imported successfully.


In [2]:
# =======================================================================================
#
# BLOCK 2: LOAD ALL PRE-TRAINED MODEL PREDICTIONS
#
# =======================================================================================
print("Loading all base model predictions from saved .npy files...")

try:
    # --- Load Mean Model Predictions ---
    oof_xgb_preds = np.load(f'{PREDS_SAVE_PATH}oof_xgb_preds.npy')
    test_xgb_preds = np.load(f'{PREDS_SAVE_PATH}test_xgb_preds.npy')
    oof_cb_preds = np.load(f'{PREDS_SAVE_PATH}oof_cb_preds.npy')
    test_cb_preds = np.load(f'{PREDS_SAVE_PATH}test_cb_preds.npy')
    oof_nn_preds = np.load(f'{PREDS_SAVE_PATH}oof_nn_preds.npy')
    test_nn_preds = np.load(f'{PREDS_SAVE_PATH}test_nn_preds.npy')
    print("  - All MEAN model predictions loaded.")

    # --- Load Error Model Predictions ---
    oof_error_preds_xgb = np.load(f'{PREDS_SAVE_PATH}oof_error_preds_xgb.npy')
    test_error_preds_xgb = np.load(f'{PREDS_SAVE_PATH}test_error_preds_xgb.npy')
    oof_error_preds_cb = np.load(f'{PREDS_SAVE_PATH}oof_error_preds_cb.npy')
    test_error_preds_cb = np.load(f'{PREDS_SAVE_PATH}test_error_preds_cb.npy')
    print("  - All ERROR model predictions loaded.")
    
    # --- Load Ground Truth Target ---
    y_true = pd.read_csv(DATA_PATH + 'dataset.csv')['sale_price']
    
except FileNotFoundError as e:
    print(f"\nERROR: Could not find a prediction file. {e}")
    print("Please ensure you have run all training notebooks and saved their predictions first.")

print("\nAll predictions loaded successfully. Ready to build the final ensemble.")

Loading all base model predictions from saved .npy files...
  - All MEAN model predictions loaded.
  - All ERROR model predictions loaded.

All predictions loaded successfully. Ready to build the final ensemble.


In [3]:
# =======================================================================================
#
# BLOCK 3: BUILD AND EVALUATE THE STAGE 1 (MEAN) ENSEMBLE
#
# =======================================================================================
print("\n--- Finding optimal weights for the 3-model mean ensemble ---")
oof_preds_stack_mean = np.vstack([oof_xgb_preds, oof_cb_preds, oof_nn_preds]).T

def get_ensemble_rmse(weights):
    final_prediction = np.dot(oof_preds_stack_mean, weights)
    return np.sqrt(mean_squared_error(y_true, final_prediction))

result = minimize(get_ensemble_rmse, [1/3]*3, method='SLSQP', bounds=[(0,1)]*3, constraints=({'type': 'eq', 'fun': lambda w: 1 - sum(w)}))
best_mean_weights = result.x

# Create the final blended mean predictions
oof_ensemble_mean = np.dot(oof_preds_stack_mean, best_mean_weights)
test_ensemble_mean = np.dot(np.vstack([test_xgb_preds, test_cb_preds, test_nn_preds]).T, best_mean_weights)

print("\n--- STAGE 1 (MEAN) ENSEMBLE RESULTS ---")
print(f"Optimal Weights (XGB/CB/NN): {best_mean_weights[0]:.4f} / {best_mean_weights[1]:.4f} / {best_mean_weights[2]:.4f}")
print(f"Final Ensemble Mean OOF RMSE : ${np.sqrt(mean_squared_error(y_true, oof_ensemble_mean)):,.2f}")


--- Finding optimal weights for the 3-model mean ensemble ---

--- STAGE 1 (MEAN) ENSEMBLE RESULTS ---
Optimal Weights (XGB/CB/NN): 0.4100 / 0.4779 / 0.1121
Final Ensemble Mean OOF RMSE : $96,778.10


In [4]:
# =======================================================================================
#
# BLOCK 4: BUILD AND EVALUATE THE STAGE 2 (ERROR) ENSEMBLE
#
# =======================================================================================
print("\n--- Blending the two champion error models (XGBoost & CatBoost) ---")
# The error target is based on the Stage 1 ensemble's mistakes
error_target_ensemble = np.abs(y_true - oof_ensemble_mean)

# We will start with a simple 50/50 average since their performance was so close.
# This is a robust choice that avoids overfitting a tiny performance difference.
oof_error_preds_final_ensemble = (oof_error_preds_xgb + oof_error_preds_cb) / 2
test_error_preds_final_ensemble = (test_error_preds_xgb + test_error_preds_cb) / 2

print("\n--- STAGE 2 (ERROR) ENSEMBLE RESULTS ---")
print(f"XGB Error OOF RMSE: ${np.sqrt(mean_squared_error(error_target_ensemble, oof_error_preds_xgb)):,.2f}")
print(f"CB Error OOF RMSE : ${np.sqrt(mean_squared_error(error_target_ensemble, oof_error_preds_cb)):,.2f}")
print(f"Blended Error OOF RMSE: ${np.sqrt(mean_squared_error(error_target_ensemble, oof_error_preds_final_ensemble)):,.2f}")


--- Blending the two champion error models (XGBoost & CatBoost) ---

--- STAGE 2 (ERROR) ENSEMBLE RESULTS ---
XGB Error OOF RMSE: $61,822.06
CB Error OOF RMSE : $61,719.37
Blended Error OOF RMSE: $61,554.94


In [5]:
# =======================================================================================
#
# BLOCK 5: FINAL CALIBRATION AND SHOWDOWN
#
# =======================================================================================
print("\n--- Calibrating the final ENSEMBLE OF ENSEMBLES ---")

oof_error_final_ensemble = np.clip(oof_error_preds_final_ensemble, 0, None)
best_score_final = float('inf')
best_a_final, best_b_final = 1.0, 1.0

# Using a more precise search grid around the previously found best values
for a in np.arange(1.90, 2.00, 0.005):
    for b in np.arange(2.15, 2.25, 0.005):
        low = oof_ensemble_mean - oof_error_final_ensemble * a
        high = oof_ensemble_mean + oof_error_final_ensemble * b
        score = winkler_score(y_true, low, high)
        if score < best_score_final:
            best_score_final = score
            best_a_final, best_b_final = a, b

print("\n" + "="*60)
print("             THE ULTIMATE PIPELINE: FINAL SHOWDOWN")
print("="*60)
print(f"Previous Best Score (CB Error Model)    : ${OLD_BEST_SCORE:,.2f}")
print(f"New ENSEMBLE-OF-ENSEMBLES Final Score   : ${best_score_final:,.2f}")
print(f"  (Using optimal multipliers a={best_a_final:.3f}, b={best_b_final:.3f})")

if best_score_final < OLD_BEST_SCORE:
    print("\nCONCLUSION: VICTORY! The Ensemble-of-Ensembles is the definitive champion!")
else:
    print("\nCONCLUSION: So close! The pipeline with the single CatBoost error model remains champion.")


--- Calibrating the final ENSEMBLE OF ENSEMBLES ---

             THE ULTIMATE PIPELINE: FINAL SHOWDOWN
Previous Best Score (CB Error Model)    : $297,534.88
New ENSEMBLE-OF-ENSEMBLES Final Score   : $296,350.26
  (Using optimal multipliers a=1.935, b=2.195)

CONCLUSION: VICTORY! The Ensemble-of-Ensembles is the definitive champion!


In [6]:
# =======================================================================================
#
# BLOCK 6: CREATE FINAL SUBMISSION FILE
#
# =======================================================================================
print("\n--- Creating the final submission file... ---")

correct_test_ids = pd.read_csv(DATA_PATH + 'test.csv', usecols=['id'])['id']
test_error_final_ensemble = np.clip(test_error_preds_final_ensemble, 0, None)

final_lower = test_ensemble_mean - test_error_final_ensemble * best_a_final
final_upper = test_ensemble_mean + test_error_final_ensemble * best_b_final
final_upper = np.maximum(final_lower, final_upper)

submission_df = pd.DataFrame({'id': correct_test_ids, 'pi_lower': final_lower, 'pi_upper': final_upper})
submission_filename = f'submission_ensemble_of_ensembles_{int(best_score_final)}.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"\n'{submission_filename}' created successfully! Good luck on the leaderboard!")
display(submission_df.head())


--- Creating the final submission file... ---

'submission_ensemble_of_ensembles_296350.csv' created successfully! Good luck on the leaderboard!


,id,pi_lower,pi_upper
0,200000,836704.565748,1.054949e+06
1,200001,572326.486963,7.936322e+05
2,200002,448132.006461,6.485328e+05
3,200003,287393.962424,4.199732e+05
4,200004,294873.474458,7.602224e+05


In [7]:
# BLOCK 5 (UPGRADED): PRECISE CALIBRATION WITH OPTIMIZER
from scipy.optimize import minimize
import numpy as np

print("\n--- Calibrating the final ENSEMBLE OF ENSEMBLES with an Optimizer ---")

# Clip the OOF error predictions to be non-negative
oof_error_final_ensemble = np.clip(oof_error_preds_final_ensemble, 0, None)

# --- Define the Objective Function for the Optimizer ---
# The function will take an array of multipliers [a, b] and return the Winkler Score.
def get_winkler_from_multipliers(multipliers):
    a, b = multipliers[0], multipliers[1]
    
    # Calculate the prediction intervals using the given multipliers
    low = oof_ensemble_mean - oof_error_final_ensemble * a
    high = oof_ensemble_mean + oof_error_final_ensemble * b
    
    # The winkler_score function is already defined in Block 1
    score = winkler_score(y_true, low, high)
    return score

# --- Run the Optimizer ---
# We provide an initial guess [a, b] and bounds to guide the search.
# Based on the grid search, [1.9, 2.2] is a good starting point.
initial_guess = [1.935, 2.195]
bounds = [(1.0, 3.0), (1.0, 3.0)] # Reasonable bounds for the multipliers

print(f"Starting optimization from initial guess: a={initial_guess[0]}, b={initial_guess[1]}")

result_calib = minimize(get_winkler_from_multipliers, 
                        initial_guess, 
                        method='L-BFGS-B', # A good method for bounded problems
                        bounds=bounds)

# Extract the best multipliers and the best score
best_a_final, best_b_final = result_calib.x
best_score_final = result_calib.fun

# --- THE ULTIMATE PIPELINE: FINAL SHOWDOWN ---
print("\n" + "="*60)
print("           THE ULTIMATE PIPELINE: FINAL SHOWDOWN")
print("="*60)
print(f"Previous Best Score (Grid Search) : ${OLD_BEST_SCORE:,.2f}")
print(f"New Score (Optimizer)             : ${best_score_final:,.2f}")
print(f" (Using optimal multipliers a={best_a_final:.4f}, b={best_b_final:.4f})")
print("="*60)

if best_score_final < OLD_BEST_SCORE:
    print("\nCONCLUSION: VICTORY! The optimizer found a better set of multipliers.")
    # You would then use these new best_a_final and best_b_final in Block 6 to create the submission
else:
    print("\nCONCLUSION: So close! The original grid search result remains the champion.")


--- Calibrating the final ENSEMBLE OF ENSEMBLES with an Optimizer ---
Starting optimization from initial guess: a=1.935, b=2.195

           THE ULTIMATE PIPELINE: FINAL SHOWDOWN
Previous Best Score (Grid Search) : $297,534.88
New Score (Optimizer)             : $296,350.03
 (Using optimal multipliers a=1.9354, b=2.1970)

CONCLUSION: VICTORY! The optimizer found a better set of multipliers.


In [8]:
# =======================================================================================
#
# BLOCK 6: CREATE FINAL SUBMISSION FILE
#
# =======================================================================================
print("\n--- Creating the final submission file... ---")

correct_test_ids = pd.read_csv(DATA_PATH + 'test.csv', usecols=['id'])['id']
test_error_final_ensemble = np.clip(test_error_preds_final_ensemble, 0, None)

final_lower = test_ensemble_mean - test_error_final_ensemble * best_a_final
final_upper = test_ensemble_mean + test_error_final_ensemble * best_b_final
final_upper = np.maximum(final_lower, final_upper)

submission_df = pd.DataFrame({'id': correct_test_ids, 'pi_lower': final_lower, 'pi_upper': final_upper})
submission_filename = f'submission_ensemble_of_ensembles_{int(best_score_final)}.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"\n'{submission_filename}' created successfully! Good luck on the leaderboard!")
display(submission_df.head())


--- Creating the final submission file... ---

'submission_ensemble_of_ensembles_296350.csv' created successfully! Good luck on the leaderboard!


,id,pi_lower,pi_upper
0,200000,836680.901599,1.055055e+06
1,200001,572302.490830,7.937398e+05
2,200002,448110.277049,6.486302e+05
3,200003,287379.586882,4.200377e+05
4,200004,294823.016778,7.604487e+05
